# American Express — Default Prediction: Exploratory Data Analysis

This notebook explores the AMEX default-prediction dataset to understand its
structure before modelling. Because the raw `train_data.csv` is **15.6 GB**,
all exploration here is done on a **sample** (the first ~1M statement rows) plus
the full 30 MB labels file.

**Key questions**
1. How balanced is the target (default vs non-default)?
2. How many monthly statements does each customer have?
3. How much data is missing, and where?
4. What are the feature groups (B/D/P/R/S)?
5. Does a key feature (`P_2`) separate the classes?

In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import config

SAMPLE_ROWS = 1_000_000

## 1. Target distribution (full labels)

In [ ]:
labels = pd.read_csv(config.TRAIN_LABELS_CSV)
print(f'Customers: {len(labels):,}')
print(f'Default rate: {labels.target.mean():.4f}')
labels.target.value_counts(normalize=True)

~458K customers with a **~26% default rate** — imbalanced but not extreme.
Note the competition metric weights the negative class ×20 to undo a 5%
down-sampling of negatives in the *public* data.

## 2. Load a sample of statements

In [ ]:
df = pd.read_csv(config.TRAIN_CSV, nrows=SAMPLE_ROWS)
df[config.DATE_COL] = pd.to_datetime(df[config.DATE_COL])
feat_cols = [c for c in df.columns if c not in config.NON_FEATURE_COLS]
print(df.shape, '|', len(feat_cols), 'features')
print('Date range:', df[config.DATE_COL].min(), '->', df[config.DATE_COL].max())
df.head()

## 3. Statements per customer

In [ ]:
spc = df.groupby(config.ID_COL).size()
print(spc.describe())
spc.value_counts().sort_index().plot(kind='bar', figsize=(7,4),
    title='Statements per customer (sample)'); plt.show()

Most customers have the **full 13 monthly statements**, but a meaningful tail
has fewer. This motivates aggregating each customer's series into fixed-length
summary features (mean/std/min/max/last) plus a `statement_count` feature.

## 4. Missingness

In [ ]:
miss = df[feat_cols].isna().mean().sort_values(ascending=False)
print('Features >50% missing:', (miss > 0.5).sum())
miss.head(15)

~30 features are >50% missing (some ~99.9%). Gradient-boosted trees handle
NaNs natively, so we keep them rather than impute.

## 5. Feature groups & categorical features

In [ ]:
groups = pd.Series([c.split('_')[0] for c in feat_cols]).value_counts()
print('Feature counts by prefix:\n', groups)
print('\nDeclared categoricals present:',
      [c for c in config.CATEGORICAL_FEATURES if c in df.columns])

Prefixes: **D**elinquency (96), **B**alance (40), **R**isk (28), **S**pend (21),
**P**ayment (3). Eleven features are categorical; `D_63`/`D_64` are strings.

## 6. Does `P_2` separate defaulters? (last statement value)

In [ ]:
last = df.sort_values(config.DATE_COL).groupby(config.ID_COL)['P_2'].last()
m = last.to_frame('P_2_last').merge(labels.set_index(config.ID_COL),
                                    left_index=True, right_index=True)
fig, ax = plt.subplots(figsize=(7,4))
for t, c in [(0,'#4C72B0'), (1,'#C44E52')]:
    ax.hist(m.loc[m.target==t,'P_2_last'].dropna(), bins=50, alpha=0.6,
            density=True, color=c, label=f'target={t}')
ax.set_title('Last P_2 by target'); ax.set_xlabel('P_2 (last)'); ax.legend(); plt.show()

`P_2` (a payment feature) clearly separates the classes — low last-`P_2`
customers default far more often. It is consistently the **single most
important feature** in the trained model (see `outputs/feature_importance.csv`).

---
## Takeaways for modelling
- Collapse each customer's ≤13 statements into summary stats (mean/std/min/max/**last**).
- `last`-value features are especially predictive (most recent state of the account).
- Keep NaNs; let LightGBM split on them.
- Optimize/early-stop on the **official AMEX metric**, not log-loss.
- See `src/feature_engineering.py` and `src/train_baseline.py` for the implementation.